In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score


In [3]:
df = pd.read_csv('../Data/final_dataset.csv')

In [4]:
y = df['target']
x = df.drop('target', axis=1) 

In [5]:
train = df[df['datetime'] < '2015-06-01']
test  = df[df['datetime'] >= '2015-06-01']

drop_cols = ['target', 'failure_flag', 'datetime', 'last_maint_datetime']

X_train = train.drop(columns=drop_cols)
y_train = train['target']

X_test = test.drop(columns=drop_cols)
y_test = test['target']

In [6]:
cat_features = ['comp', 'model']
num_features = [col for col in X_train.columns if col not in cat_features]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', 'passthrough', num_features)
    ]
)

In [7]:
rfc = RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced')
model = Pipeline([
    ('preprocess', preprocess),
    ('model', rfc)
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00    504718
           1       0.85      0.64      0.73      9606

    accuracy                           0.99    514324
   macro avg       0.92      0.82      0.86    514324
weighted avg       0.99      0.99      0.99    514324

[[503608   1110]
 [  3421   6185]]
